In [1]:
from pathlib import Path

BASE_DIR = Path().resolve().parents[1]
print(BASE_DIR)

D:\17. Hackathon\Project\flood-analysis-project


In [14]:
from pathlib import Path
import rasterio
import pandas as pd
import numpy as np

In [ ]:
BASE_DIR = Path().resolve().parents[1]

RUNOFF_PATH = BASE_DIR / "data" / "raw" / "runoff" / "runoff_points_new.csv"
RAIN_PATH = BASE_DIR / "data" / "interim"/ "rainfall" / "corrected_processed.csv"

OUTPUT_RUNOFF = BASE_DIR / "data" / "interim" /"runoff" /"runoff_delhi.csv"
OUTPUT_FINAL = BASE_DIR / "data" / "interim" / "rainfall_runoff" 

In [40]:
import pandas as pd

df = pd.read_csv(RUNOFF_PATH)

df = df.rename(columns={
    'X': 'lon',
    'Y': 'lat',
    'runoff': 'runoff_raw'
})

In [41]:
print(df)

                lon        lat  runoff_raw
0         77.081483  28.894416         0.6
1         77.081587  28.894416         0.6
2         77.081691  28.894416         0.6
3         77.081795  28.894416         0.6
4         77.081899  28.894416         0.6
...             ...        ...         ...
17581622  77.374082  28.394787         0.4
17581623  77.374186  28.394787         0.4
17581624  77.374290  28.394787         0.4
17581625  77.374394  28.394787         0.4
17581626  77.374498  28.394787         0.4

[17581627 rows x 3 columns]


In [42]:
min_val = df['runoff_raw'].min()
max_val = df['runoff_raw'].max()

df['runoff_norm'] = (
    (df['runoff_raw'] - min_val) / (max_val - min_val)
)

df['runoff_coefficient'] = 1 - df['runoff_norm']

In [43]:
print(df)

                lon        lat  runoff_raw  runoff_norm  runoff_coefficient
0         77.081483  28.894416         0.6     0.615385            0.384615
1         77.081587  28.894416         0.6     0.615385            0.384615
2         77.081691  28.894416         0.6     0.615385            0.384615
3         77.081795  28.894416         0.6     0.615385            0.384615
4         77.081899  28.894416         0.6     0.615385            0.384615
...             ...        ...         ...          ...                 ...
17581622  77.374082  28.394787         0.4     0.307692            0.692308
17581623  77.374186  28.394787         0.4     0.307692            0.692308
17581624  77.374290  28.394787         0.4     0.307692            0.692308
17581625  77.374394  28.394787         0.4     0.307692            0.692308
17581626  77.374498  28.394787         0.4     0.307692            0.692308

[17581627 rows x 5 columns]


In [44]:
print(df.columns)

Index(['lon', 'lat', 'runoff_raw', 'runoff_norm', 'runoff_coefficient'], dtype='str')


In [45]:
pd.set_option('display.max_columns', None)
print(df.head())

         lon        lat  runoff_raw  runoff_norm  runoff_coefficient
0  77.081483  28.894416         0.6     0.615385            0.384615
1  77.081587  28.894416         0.6     0.615385            0.384615
2  77.081691  28.894416         0.6     0.615385            0.384615
3  77.081795  28.894416         0.6     0.615385            0.384615
4  77.081899  28.894416         0.6     0.615385            0.384615


In [46]:
rain_df = pd.read_csv(RAIN_PATH)

# fix precision mismatch
df['lat'] = df['lat'].round(4)
df['lon'] = df['lon'].round(4)

rain_df['lat'] = rain_df['lat'].round(4)
rain_df['lon'] = rain_df['lon'].round(4)

final_df = rain_df.merge(
    df[['lat','lon','runoff_coefficient']],
    on=['lat','lon'],
    how='left'
)

In [48]:
final_df['runoff'] = (
    final_df['rain_corrected'] *
    final_df['runoff_coefficient']
)

In [49]:
print(final_df)

          lat    lon        date  rain_gpm  imd_interp  rain_corrected  \
0       26.15  76.85  2015-01-01      0.91         0.0            0.91   
1       26.25  76.85  2015-01-01      0.17         0.0            0.17   
2       26.35  76.85  2015-01-01      0.06         0.0            0.06   
3       26.45  76.85  2015-01-01      0.07         0.0            0.07   
4       26.55  76.85  2015-01-01      0.00         0.0            0.00   
...       ...    ...         ...       ...         ...             ...   
346290  28.15  77.35  2026-03-16      0.00         0.0            0.00   
346291  28.65  77.35  2026-03-16      0.00         0.0            0.00   
346292  28.75  77.35  2026-03-16      0.00         0.0            0.00   
346293  29.25  77.35  2026-03-16      0.00         0.0            0.00   
346294  29.45  77.35  2026-03-16      0.00         0.0            0.00   

        runoff_coefficient  runoff  
0                      NaN     NaN  
1                      NaN     NaN  


In [51]:
final_df.to_csv("final_with_runoff.csv", index=False)

In [52]:
OUTPUT_FINAL = BASE_DIR / "data" / "interim" / "rainfall_runoff" 

In [54]:
print(OUTPUT_FINAL)

D:\17. Hackathon\Project\flood-analysis-project\data\interim\rainfall_runoff


In [53]:
final_df.to_csv(OUTPUT_FINAL / "final_with_runoff.csv", index=False)